# Notebook: Backup & Restore Power BI Large Semantic Models Across Regions

## Prerequisites
- Workspace-level storage permissions must be enabled in **Admin Portal > Azure Connections**.
- The storage account must already exist and be linked to a workspace or tenant.
- Review [Microsoft's limitations](https://learn.microsoft.com/en-us/power-bi/enterprise/service-premium-backup-restore-dataset#considerations-and-limitations).


## Step 0: Setup

In [1]:
%pip install semantic-link-labs

StatementMeta(, 238c6b63-05e1-4641-b2fb-312e1eefeafb, 7, Finished, Available, Finished, True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.0/888.0 kB 8.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 44.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 51.9 MB/s eta 0:00:00
  Attempting uninstall: azure-core
    Found existing installation: azure-core 2024.9.1
    Not uninstalling azure-core at /home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages, outside environment /nfs4/pyenv-93e0bdb0-3212-48fa-bd0b-943ee6b1fb1e
    Can't uninstall 'azure-core'. No files were found to uninstall.
  At

In [3]:
import sempy_labs as labs
import sempy_labs.admin as labs_admin
import sempy.fabric as fabric

StatementMeta(, 238c6b63-05e1-4641-b2fb-312e1eefeafb, 10, Finished, Available, Finished, False)

In [4]:
# Configure your variables

# Storage account name (must already exist and be linked to workspace/tenant)
storage_account_name = "storagecem0011"

# Source workspace name where your large semantic models are located
workspace = "test_move_1"

# Target capacity name in the destination region
target_capacity = "fcuaen64"

print(f"Storage Account: {storage_account_name}")
print(f"Workspace ID: {workspace}")
print(f"Target Capacity: {target_capacity}")
print("\n⚠️  Please ensure you have updated these values before proceeding with the migration!")

StatementMeta(, 238c6b63-05e1-4641-b2fb-312e1eefeafb, 11, Finished, Available, Finished, False)

Storage Account: storagecem0011
Workspace ID: test_move_1
Target Capacity: fcuaen64

⚠️  Please ensure you have updated these values before proceeding with the migration!


## Step 1: Backup Semantic Models

In [18]:
import pandas as pd
from sempy import fabric
import sempy_labs as labs
import sempy_labs.admin as labs_admin

# =========================
# Parameters
# =========================
workspace = "test_move_1"

# Empty list = all semantic models in workspace
# Example:
# semantic_models = []
# semantic_models = ["Sales Model"]
# semantic_models = ["Sales Model", "Finance Model"]
semantic_models = ["customerorders"]

# Set to True only if you really need to attach the workspace
# to the ADLS/dataflow storage account for the first time.
# Normally keep this False for regular backup runs.
assign_storage = False
storage_account_name = "storagecem0011"


# =========================
# Resolve workspace
# =========================
workspace_id = fabric.resolve_workspace_id(workspace=workspace)

# Optional: assign workspace to dataflow storage only once
if assign_storage:
    labs.assign_workspace_to_dataflow_storage(
        dataflow_storage_account=storage_account_name,
        workspace=workspace_id
    )

# =========================
# Get all semantic models in workspace
# =========================
df_models = labs_admin.list_datasets()

df_workspace_models = df_models[df_models["Workspace Id"] == workspace_id].copy()

# Optional: keep only import models if you want
# Uncomment if needed and if column exists in your environment
# df_workspace_models = df_workspace_models[df_workspace_models["Configured By"].notna()]

# =========================
# Filter by semantic model list if provided
# =========================
if semantic_models:
    requested_models_lower = {m.strip().lower() for m in semantic_models}
    df_workspace_models = df_workspace_models[
        df_workspace_models["Dataset Name"].str.strip().str.lower().isin(requested_models_lower)
    ].copy()

    found_models_lower = set(df_workspace_models["Dataset Name"].str.strip().str.lower())
    missing_models = [m for m in semantic_models if m.strip().lower() not in found_models_lower]

    if missing_models:
        print("These semantic models were not found in the workspace:")
        for m in missing_models:
            print(f" - {m}")

# =========================
# Validate
# =========================
if df_workspace_models.empty:
    raise ValueError("No semantic models found to process.")

print("Semantic models selected for backup:")
for model_name in df_workspace_models["Dataset Name"].tolist():
    print(f" - {model_name}")

# =========================
# Backup each semantic model
# Works for both Small and Large storage formats
# =========================
results = []

for _, row in df_workspace_models.iterrows():
    dataset_name = row["Dataset Name"]

    try:
        print(f"\nProcessing semantic model: {dataset_name}")

        # Backup semantic model to ABF
        labs.backup_semantic_model(
            dataset=dataset_name,
            file_path=f"{dataset_name}.abf",
            allow_overwrite=True,
            apply_compression=True,
            workspace=workspace_id
        )

        results.append({
            "Semantic Model": dataset_name,
            "Status": "Success",
            "Message": "Backup completed"
        })

        print(f"Backup completed for: {dataset_name}")

    except Exception as e:
        results.append({
            "Semantic Model": dataset_name,
            "Status": "Failed",
            "Message": str(e)
        })

        print(f"Backup failed for: {dataset_name}")
        print(str(e))

# =========================
# Summary
# =========================
df_results = pd.DataFrame(results)
display(df_results)

StatementMeta(, 238c6b63-05e1-4641-b2fb-312e1eefeafb, 25, Finished, Available, Finished, False)

Semantic models selected for backup:
 - customerorders

Processing semantic model: customerorders
🟢 The 'customerorders' semantic model within the 'test_move_1' workspace has been backed up to the 'customerorders.abf' location.
Backup completed for: customerorders


SynapseWidget(Synapse.DataFrame, 7d2f5d6a-9d32-47e3-8660-1a617c54fda5)

## Step 2: Migrate Workspace

In [ ]:
labs.assign_workspace_to_capacity(
    capacity=target_capacity,
    workspace=workspace_id
)


## Step 3: Restore Semantic Models

In [10]:
# Set format to Large
for _, row in df_large_models.iterrows():
    name = row['Dataset Name']
    labs.set_semantic_model_storage_format(dataset=name, storage_format='Large', workspace=workspace_id)

# Wait for format update (manual delay or retry logic may be needed)

# Restore models
for _, row in df_large_models.iterrows():
    name = row['Dataset Name']
    labs.restore_semantic_model(
        dataset=name,
        file_path=f"{name}.abf",
        allow_overwrite=True,
        workspace=workspace_id,
        force_restore=True
    )


StatementMeta(, 943da2d9-ca21-4dc2-a95c-e6f31c6d14dd, 17, Finished, Available, Finished, False)

NameError: name 'df_large_models' is not defined

## Notes
- Add a delay or retry logic between format change and restore.
- Re-run initialization cells if the session expires.

## Resources
- Blog Post - https://davidmitchell.dev/how-to-migrate-large-power-bi-semantic-models-across-regions-without-breaking-reports/

- Large Semantic Model Format - https://learn.microsoft.com/en-us/power-bi/enterprise/service-premium-large-models
- Configure dataflow storage to use Azure Data Lake Gen 2 - https://learn.microsoft.com/en-us/power-bi/transform-model/dataflows/dataflows-azure-data-lake-storage-integration

## Video Walkthrough

<iframe width="560" height="315" src="https://www.youtube.com/embed/phlAVzTGEG0" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" allowfullscreen></iframe>